## Gene Ontology Analyses of `Sox4` Targets

## Purpose: 

For each parameter combination, take all `Sox4` target genes predicted from `JASPAR` and run `GO` analysis.
Output tables and summary results. 

## Packages and Options

In [1]:
import networkx, pickle, glob, os

from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

from goatools.obo_parser import GODag
from goatools.goea.go_enrichment_ns import GOEnrichmentStudyNS
from goatools.base import download_go_basic_obo

import pandas as pd
pd.set_option('display.max_rows', 1000000)
pd.set_option('display.max_columns', 1000000)
pd.set_option('display.max_colwidth', 10000000)

## Load Gene Lists 

Load lists of genes that are `Sox4` targets per each parameter combination

In [2]:
sox4_target_dict = {}

for file in glob.glob("../../../../outputs/pickled_networks/differential_expression_tf_networks/*"): 
    
    network = pickle.load(open(file, 'rb'))
    
    sox4_target_dict[file.split("/")[-1].split(".")[0]] =  [edge_pair[1] for edge_pair in network.out_edges(["sox4"])]


## Setup for `GOATOOLS` (Tool for `GO` Analysis)

In [3]:
download_go_basic_obo()

  EXISTS: go-basic.obo


'go-basic.obo'

In [4]:
# file for population (protein-coding genes)
protein_coding_genes = []

with open("../../../../inputs/goatools_input/population.txt", 'r') as in_file: 
    for line in in_file: 
        protein_coding_genes.append(line.strip("\n"))
        
# file for gene ontology associations
associations = pickle.load(open("../../../../inputs/goatools_input/mouse/gene_ontology_associations.pkl", 'rb'))

## Run `GO` Analysis

In [5]:
for param_combo in sox4_target_dict: 
    
    # initialize object with population, associations, and GO DAG
    enrichment_analysis_obj = GOEnrichmentStudyNS(

        protein_coding_genes,
        associations, 
        GODag("go-basic.obo"), 

        # not propagating counts to parents of GO terms 
        propagate_counts = False, 
        methods = ["fdr_bh"]
    )
    
    results = enrichment_analysis_obj.run_study(
        sox4_target_dict[param_combo]
    )
    
    enrichment_analysis_obj.wr_tsv(param_combo, results)

# remove the OBO file
os.remove("go-basic.obo")

go-basic.obo: fmt(1.2) rel(2024-01-17) 45,869 Terms

Load BP Ontology Enrichment Analysis ...
 85% 18,620 of 21,965 population items found in association

Load CC Ontology Enrichment Analysis ...
 85% 18,761 of 21,965 population items found in association

Load MF Ontology Enrichment Analysis ...
 85% 18,703 of 21,965 population items found in association

Runing BP Ontology Analysis: current study set of 81 IDs.
 98%     79 of     81 study items found in association
100%     81 of     81 study items found in population(21965)
Calculating 12,503 uncorrected p-values using fisher_scipy_stats
  12,503 terms are associated with 18,612 of 21,965 population items
   1,094 terms are associated with     79 of     81 study items
  METHOD fdr_bh:
      11 GO terms found significant (< 0.05=alpha) ( 11 enriched +   0 purified): statsmodels fdr_bh
      25 study items associated with significant GO IDs (enriched)
       0 study items associated with significant GO IDs (purified)

Runing CC Ontolo

## Concatenate All Significant Results Across Parameter Iterations

In [6]:
concat_df = []

for file in glob.glob("upstream_*"): 
    tmp_df = pd.read_csv(file, sep="\t")
    
    tmp_df["Param Combo"] = file
    
    concat_df.append(tmp_df[(tmp_df["p_fdr_bh"]<0.05) & (tmp_df["study_count"]>=4)])
    
    os.remove(file)

concat_df = pd.concat(concat_df)
concat_df.head()

,# GO,NS,enrichment,name,ratio_in_study,ratio_in_pop,p_uncorrected,depth,study_count,p_fdr_bh,study_items,Param Combo
0,GO:0008203,BP,e,cholesterol metabolic process,22/284,116/21965,1.177710e-19,6,22,1.472491e-15,"abca1, apoa1, apoa2, apob, apoe, cubn, cyp27a1, cyp51, dhcr24, dhcr7, fdft1, fdx1, hmgcs1, hnf1a, insig1, ldlr, mttp, mvk, nsdhl, sqle, srebf2, tm7sf2",upstream_750_score_900
1,GO:0008202,BP,e,steroid metabolic process,18/284,127/21965,5.479940e-14,4,18,3.425784e-10,"akr1c19, apoa1, apob, cubn, cyp27a1, cyp51, dhcr24, dhcr7, fdft1, fdx1, hmgcs1, insig1, ldlr, mttp, mvk, nsdhl, srebf2, tm7sf2",upstream_750_score_900
2,GO:0042632,BP,e,cholesterol homeostasis,16/284,99/21965,1.727625e-13,5,16,7.200165e-10,"abca1, apoa1, apoa2, apob, apoe, fabp4, fgfr4, gramd1b, hnf4a, insig1, ldlr, lpl, mttp, nr5a2, rora, srebf2",upstream_750_score_900
3,GO:0016126,BP,e,sterol biosynthetic process,10/284,29/21965,1.797797e-12,6,10,5.619465e-09,"cyp51, dhcr24, dhcr7, fdft1, hmgcs1, insig1, mvk, nsdhl, sqle, tm7sf2",upstream_750_score_900
4,GO:0006629,BP,e,lipid metabolic process,35/284,714/21965,1.423814e-11,3,35,3.560390e-08,"acly, aldh1a1, apoa1, apob, apoe, b4galt6, cubn, cyp26b1, cyp27a1, cyp51, dhcr24, dhcr7, elovl5, elovl6, fads2, fasn, fdft1, fdx1, gstm1, hmgcs1, hnf4a, insig1, ldlr, lpl, mttp, mvk, nsdhl, pip4k2b, plcd3, prkag3, scd2, sqle, srebf2, tecrl, tm7sf2",upstream_750_score_900


## Output Data

`GO` Enrichments

In [7]:
# GO enrichments
concat_df.to_csv("../outputs/sox4_target_genes_GO_enrichment_fdr_0.05_input_gene_greater_equal_4.csv", index=False)

Frequency of `GO` Terms

In [8]:
frequency = pd.DataFrame(concat_df[["name"]].value_counts()).reset_index()
frequency.columns=["GO Name", "Found in # of parameter combinations (OUT OF 9)"]

frequency.to_csv("../outputs/sox4_target_genes_most_common_enriched_pathways_fdr_0.05_input_gene_greater_equal_4.csv", index=False)

frequency.head()

,GO Name,Found in # of parameter combinations (OUT OF 9)
0,cholesterol biosynthetic process,7
1,steroid metabolic process,7
2,cholesterol homeostasis,7
3,axon,7
4,endoplasmic reticulum,7


`Sox4` Target Gene Lists 

In [9]:
concat_sox4_target_list = []

for param_combo in sox4_target_dict: 
    tmp_df = pd.DataFrame()
    tmp_df["Genes"] = sox4_target_dict[param_combo] 
    tmp_df["Parameter Combination"] = param_combo
    
    concat_sox4_target_list.append(tmp_df)

concat_sox4_target_list = pd.concat(concat_sox4_target_list).sort_values(by=["Parameter Combination"])

concat_sox4_target_list.to_csv("../outputs/sox4_target_genes.csv", index=False)

concat_sox4_target_list

,Genes,Parameter Combination
308,dchs2,upstream_2500_score_900
129,svep1,upstream_2500_score_900
128,tnc,upstream_2500_score_900
127,ccn1,upstream_2500_score_900
126,mttp,upstream_2500_score_900
125,slc35a3,upstream_2500_score_900
124,crct1,upstream_2500_score_900
123,tm4sf1,upstream_2500_score_900
122,car3,upstream_2500_score_900
121,gnas,upstream_2500_score_900
